# Progression Quality: Turning Territory into Line Breaks

## Purpose
This notebook extends the Field Tilt Proxy and Champion Profile analyses by asking whether territorial control translated into line-breaking progression.

## Research Question
Did teams that controlled more final-third territory also progress through defensive lines more effectively?

## Data Scope
- Source file: `site_official_stats_team_wide_flagged.csv`
- Source: FIFA official Match Centre statistics
- Main filter: `stats_complete == True`
- Excluded match: Belgium vs Egypt (`match_id = 400021478`), because FIFA only provides Live Statistics for that match
- Views: Overall, Group Stage, Knockout Stage

## Key Metrics
- Field Tilt Proxy: team final-third entries / both teams' final-third entries x 100
- Line Break Completion Rate: completed line breaks / attempted line breaks x 100
- Defensive Line Break Completion Rate: completed defensive line breaks / attempted defensive line breaks x 100
- Completed Defensive Line Breaks per Match: completed defensive line breaks / matches played

## Interpretation Rule
Field Tilt Proxy describes territory. Line-break metrics describe progression quality. A team can enter the final third often without consistently breaking defensive lines.

## Data Limitations for Publication
- Team match counts differ across the tournament because some teams played 3 matches and finalists played up to 8 matches. Spain-centered rankings are descriptive champion profiling, not a causal model of why Spain won.
- Belgium vs Egypt (`match_id = 400021478`) is excluded from the main analytical tables because FIFA provides only `Live Statistics` for that match. As a result, Belgium has 5 full-stat matches instead of 6, and Egypt has 4 full-stat matches instead of 5. Per-match metrics for those two teams can be slightly inflated because one real match is not in the denominator.




In [ ]:
"""
Step 1: Environment setup and data load
=======================================
Only BASE_DIR should need editing if the project folder moves.
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

BASE_DIR = Path.cwd().resolve()
for parent in [BASE_DIR, *BASE_DIR.parents]:
    if parent.name == "worldcup-2026-official-stats-analysis":
        BASE_DIR = parent
        break
DATA_DIR = BASE_DIR / "data" / "fifa_worldcup_2026" / "site_scrape"
PUBLIC_DIR = BASE_DIR
OUTPUT_DATA_DIR = PUBLIC_DIR / "data"
OUTPUT_FIG_DIR = PUBLIC_DIR / "figures"

RAW_CSV_PATH = DATA_DIR / "site_official_stats_team_wide_flagged.csv"
GROUP_STAGE_PATH = DATA_DIR / "site_official_stats_team_wide_group_stage.csv"

for path_name, path_value in {
    "BASE_DIR": BASE_DIR,
    "DATA_DIR": DATA_DIR,
    "PUBLIC_DIR": PUBLIC_DIR,
    "RAW_CSV_PATH": RAW_CSV_PATH,
    "GROUP_STAGE_PATH": GROUP_STAGE_PATH,
}.items():
    if not path_value.exists():
        raise FileNotFoundError(f"{path_name} does not exist: {path_value}")

OUTPUT_DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FIG_DIR.mkdir(parents=True, exist_ok=True)

raw = pd.read_csv(RAW_CSV_PATH)

print("[Raw file]")
print(f"Rows: {len(raw):,}")
print(f"Matches: {raw['match_id'].nunique():,}")
print(f"Teams: {raw['team_name'].nunique():,}")

display(
    raw[["match_id", "stats_source", "stats_complete"]]
    .drop_duplicates()
    .groupby(["stats_source", "stats_complete"])
    .size()
    .reset_index(name="matches")
)

# Keep only full FIFA Official Stats matches. Missing values are not imputed.
df = raw[raw["stats_complete"] == True].copy()

print("\n[Analysis file]")
print(f"Rows: {len(df):,}")
print(f"Matches: {df['match_id'].nunique():,}")
print(f"Teams: {df['team_name'].nunique():,}")




## Build Team-Level Progression Metrics

This step converts match-team rows into team-level indicators by phase. Rate metrics are calculated from phase totals, not by averaging match-level rates.

This matters because teams played different numbers of matches in the knockout stage. Per-match and rate metrics reduce the bias from raw totals.


In [ ]:
"""
Step 2: Build team-level progression metrics by phase
=====================================================
"""

CHAMPION_TEAM = "Spain"

group_stage_match_ids = set(pd.read_csv(GROUP_STAGE_PATH)["match_id"].unique())

df["competition_phase"] = np.where(
    df["match_id"].isin(group_stage_match_ids),
    "Group Stage",
    "Knockout Stage"
)

print("[Phase split]")
display(
    df[["match_id", "competition_phase"]]
    .drop_duplicates()
    .groupby("competition_phase")
    .size()
    .reset_index(name="matches")
)

final_third_cols = [
    "attacking__final_third_entries__left_channel",
    "attacking__final_third_entries__left_inside_channel",
    "attacking__final_third_entries__central_channel",
    "attacking__final_third_entries__right_inside_channel",
    "attacking__final_third_entries__right_channel",
]

required_cols = final_third_cols + [
    "match_id",
    "team_name",
    "team_side",
    "opponent_side",
    "attacking__line_breaks__attempted_line_breaks",
    "attacking__line_breaks__completed_line_breaks",
    "attacking__line_breaks__attempted_defensive_line_breaks",
    "attacking__line_breaks__completed_defensive_line_breaks",
    "attacking__attempts_at_goal__total",
    "attacking__goal__total",
]

missing_required = [col for col in required_cols if col not in df.columns]
if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")


def safe_divide(numerator, denominator):
    return np.where(denominator == 0, np.nan, numerator / denominator)


def calculate_progression_metrics(input_df, phase_label):
    temp = input_df.copy()

    temp["final_third_entries_total"] = temp[final_third_cols].sum(axis=1, min_count=5)

    opponent_f3 = (
        temp[["match_id", "team_side", "final_third_entries_total"]]
        .rename(columns={
            "team_side": "opponent_side",
            "final_third_entries_total": "opponent_final_third_entries_total",
        })
    )

    temp = temp.merge(
        opponent_f3,
        on=["match_id", "opponent_side"],
        how="left",
        validate="many_to_one",
    )

    valid = temp.dropna(subset=["final_third_entries_total", "opponent_final_third_entries_total"]).copy()

    team = (
        valid
        .groupby("team_name")
        .agg(
            matches=("match_id", "nunique"),
            final_third_entries_total=("final_third_entries_total", "sum"),
            opponent_final_third_entries_total=("opponent_final_third_entries_total", "sum"),
            attempted_line_breaks=("attacking__line_breaks__attempted_line_breaks", "sum"),
            completed_line_breaks=("attacking__line_breaks__completed_line_breaks", "sum"),
            attempted_defensive_line_breaks=("attacking__line_breaks__attempted_defensive_line_breaks", "sum"),
            completed_defensive_line_breaks=("attacking__line_breaks__completed_defensive_line_breaks", "sum"),
            attempts_at_goal=("attacking__attempts_at_goal__total", "sum"),
            goals=("attacking__goal__total", "sum"),
        )
        .reset_index()
    )

    team["competition_phase"] = phase_label

    team["field_tilt_proxy"] = safe_divide(
        team["final_third_entries_total"],
        team["final_third_entries_total"] + team["opponent_final_third_entries_total"]
    ) * 100

    team["final_third_entries_per_match"] = safe_divide(team["final_third_entries_total"], team["matches"])
    team["line_break_completion_rate"] = safe_divide(team["completed_line_breaks"], team["attempted_line_breaks"]) * 100
    team["defensive_line_break_completion_rate"] = safe_divide(
        team["completed_defensive_line_breaks"],
        team["attempted_defensive_line_breaks"]
    ) * 100
    team["completed_line_breaks_per_match"] = safe_divide(team["completed_line_breaks"], team["matches"])
    team["completed_defensive_line_breaks_per_match"] = safe_divide(team["completed_defensive_line_breaks"], team["matches"])
    team["defensive_line_break_share"] = safe_divide(
        team["completed_defensive_line_breaks"],
        team["completed_line_breaks"]
    ) * 100
    team["shot_creation_efficiency"] = safe_divide(team["attempts_at_goal"], team["final_third_entries_total"]) * 100
    team["goals_per_attempt"] = safe_divide(team["goals"], team["attempts_at_goal"]) * 100

    return team


progression_overall = calculate_progression_metrics(df, "Overall")
progression_group_stage = calculate_progression_metrics(df[df["competition_phase"] == "Group Stage"], "Group Stage")
progression_knockout_stage = calculate_progression_metrics(df[df["competition_phase"] == "Knockout Stage"], "Knockout Stage")

progression_metrics_by_phase = pd.concat(
    [progression_overall, progression_group_stage, progression_knockout_stage],
    ignore_index=True,
)

progression_metrics_path = OUTPUT_DATA_DIR / "progression_quality_team_metrics_by_phase.csv"
progression_metrics_by_phase.to_csv(progression_metrics_path, index=False, encoding="utf-8-sig")

print(f"[Check] Saved progression metrics: {progression_metrics_path}")
display(
    progression_metrics_by_phase
    .groupby("competition_phase")
    .agg(
        teams=("team_name", "nunique"),
        avg_matches=("matches", "mean"),
        min_matches=("matches", "min"),
        max_matches=("matches", "max"),
        avg_field_tilt=("field_tilt_proxy", "mean"),
        avg_lb_completion=("line_break_completion_rate", "mean"),
        avg_def_lb_completion=("defensive_line_break_completion_rate", "mean"),
    )
    .round(2)
    .reset_index()
)




## Ranking Context

Before plotting, this table shows which teams ranked highest in territory and line-breaking progression. It helps identify whether the champion profile was unusual or part of a wider tournament pattern.


In [ ]:
"""
Step 3: Ranking context for key progression metrics
===================================================
"""

rank_phase = "Overall"
# rank_phase = "Group Stage"
# rank_phase = "Knockout Stage"

rank_df = progression_metrics_by_phase[progression_metrics_by_phase["competition_phase"] == rank_phase].copy()

ranking_metrics = [
    "field_tilt_proxy",
    "line_break_completion_rate",
    "defensive_line_break_completion_rate",
    "completed_line_breaks_per_match",
    "completed_defensive_line_breaks_per_match",
]

for metric in ranking_metrics:
    rank_df[f"{metric}_rank"] = rank_df[metric].rank(ascending=False, method="min").astype(int)

ranking_view = rank_df[
    ["team_name", "matches"]
    + ranking_metrics
    + [f"{metric}_rank" for metric in ranking_metrics]
].copy()

ranking_view = ranking_view.sort_values("field_tilt_proxy_rank")

ranking_path = OUTPUT_DATA_DIR / f"progression_quality_rankings_{rank_phase.lower().replace(' ', '_')}.csv"
ranking_view.to_csv(ranking_path, index=False, encoding="utf-8-sig")

print(f"Saved rankings: {ranking_path}")
display(ranking_view.head(12).round(2))

champion_context = ranking_view[ranking_view["team_name"] == CHAMPION_TEAM].copy()
print(f"\n{CHAMPION_TEAM} ranking context:")
display(champion_context.round(2))




## Visualization 1: Territory vs Line Break Completion

This chart tests whether territory control aligned with general line-breaking efficiency.

- Right side: teams with stronger final-third territory control
- Higher position: teams with better line-break completion rate
- Spain is highlighted as the champion benchmark


In [ ]:
 """
Step 4: Plot Field Tilt Proxy vs Line Break Completion Rate
===========================================================
Updated: point size AND color both reflect matches played, using a
categorical palette (5 distinct hues). Semifinalists are identified by
color alone (8 matches = red, unique to them) — no separate ring needed,
since the color category already isolates them uniquely.
"""

import sys
import subprocess
try:
    from adjustText import adjust_text
except ModuleNotFoundError:
    print("[Setup] adjustText is not installed in this Jupyter kernel. Installing now...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "adjustText"])
    from adjustText import adjust_text
    print("[Setup] adjustText installed successfully.")

DISPLAY_NAME_MAP = {
    "Curaçao": "Curacao",
    "Côte d’Ivoire": "Cote d'Ivoire",
    "Côte d'Ivoire": "Cote d'Ivoire",
    "Türkiye": "Turkiye",
    "Turkey": "Turkiye",
    "IR Iran": "Iran",
}

SEMIFINALISTS = ["Spain", "Argentina", "France", "England"]

MATCH_COLOR_PALETTE = ["#8b5cf6", "#3b82f6", "#10b981", "#f59e0b", "#dc2626"]


def display_team_name(team):
    return DISPLAY_NAME_MAP.get(team, team)


def plot_progression_scatter(metric_y, y_label, title_suffix, output_suffix, phase_label="Overall"):
    plot_df = progression_metrics_by_phase[progression_metrics_by_phase["competition_phase"] == phase_label].copy()
    plot_df = plot_df.dropna(subset=["field_tilt_proxy", metric_y])

    if plot_df.empty:
        raise ValueError(f"No data available for {phase_label} and {metric_y}")

    missing_sf = [t for t in SEMIFINALISTS if t not in plot_df["team_name"].values]
    if missing_sf:
        raise ValueError(f"Semifinalist teams missing from data: {missing_sf}")

    x_median = plot_df["field_tilt_proxy"].median()
    y_median = plot_df[metric_y].median()

    fig, ax = plt.subplots(figsize=(13, 9), dpi=150)
    fig.patch.set_facecolor("#fbfaf7")
    ax.set_facecolor("#fbfaf7")

    ax.axvspan(plot_df["field_tilt_proxy"].min() - 4, x_median, color="#f3f4f6", alpha=0.55, zorder=0)
    ax.axvspan(x_median, plot_df["field_tilt_proxy"].max() + 4, color="#ecfdf5", alpha=0.38, zorder=0)

    min_m, max_m = plot_df["matches"].min(), plot_df["matches"].max()
    size_min, size_max = 38, 145
    plot_df["point_size"] = size_min + (plot_df["matches"] - min_m) / (max_m - min_m) * (size_max - size_min)

    match_values = sorted(plot_df["matches"].unique())
    if len(match_values) > len(MATCH_COLOR_PALETTE):
        raise ValueError(
            f"MATCH_COLOR_PALETTE has {len(MATCH_COLOR_PALETTE)} colors but "
            f"{len(match_values)} distinct match counts were found: {match_values}"
        )
    color_map = dict(zip(match_values, MATCH_COLOR_PALETTE))
    plot_df["point_color"] = plot_df["matches"].map(color_map)

    is_semifinalist = plot_df["team_name"].isin(SEMIFINALISTS)

    # Semifinalists are already uniquely identified by the 8-match color, so no separate outline ring is added
    ax.scatter(
        plot_df["field_tilt_proxy"],
        plot_df[metric_y],
        s=plot_df["point_size"],
        c=plot_df["point_color"],
        edgecolors="white",
        linewidth=0.75,
        alpha=0.9,
        zorder=3,
    )

    texts = []
    for _, row in plot_df.iterrows():
        is_sf = row["team_name"] in SEMIFINALISTS
        texts.append(
            ax.text(
                row["field_tilt_proxy"],
                row[metric_y],
                display_team_name(row["team_name"]),
                fontsize=8.5 if is_sf else 7,
                fontweight="bold" if is_sf else "normal",
                color="#111827",
                zorder=6 if is_sf else 4,
            )
        )

    adjust_text(
        texts,
        ax=ax,
        expand=(1.7, 2.0),
        force_text=(1.0, 1.3),
        force_static=(0.6, 0.8),
        force_pull=(0.008, 0.008),
        max_move=(70, 70),
        min_arrow_len=3,
        arrowprops=dict(arrowstyle="-", color="#475569", lw=0.7, alpha=0.8),
        iter_lim=4000,
    )

    ax.axvline(x_median, linestyle="--", color="#6b7280", linewidth=1.0, alpha=0.80)
    ax.axhline(y_median, linestyle="--", color="#6b7280", linewidth=1.0, alpha=0.80)

    ax.set_xlabel("Field Tilt Proxy (%)", fontsize=11)
    ax.set_ylabel(y_label, fontsize=11)
    ax.set_title(f"Territory vs Progression Quality: {title_suffix} ({phase_label})", fontsize=15, pad=14)

    ax.grid(alpha=0.18, color="#9ca3af", linewidth=0.8)
    ax.set_axisbelow(True)
    for spine in ax.spines.values():
        spine.set_visible(False)

    from matplotlib.lines import Line2D
    legend_handles = [
        Line2D([0], [0], marker="o", color="w", markerfacecolor=color_map[c], markeredgecolor="white",
               markersize=10, label=f"{c} matches" + (" (Semifinalists)" if c == max_m else ""))
        for c in match_values
    ]
    ax.legend(handles=legend_handles, loc="lower right", fontsize=8.5, frameon=True,
              facecolor="#fbfaf7", edgecolor="#d1d5db", title="Matches played", title_fontsize=8.5)

    footnote = (
        "Point size and color both reflect matches played (categorical palette). Semifinalists uniquely identified by the 8-match color (red).\n"
        "Field Tilt Proxy measures final-third territory share; line-break metrics describe progression quality.\n"
        "Source: FIFA Match Centre | Full Official Stats only.\nNote: descriptive rankings; match counts 3-8; Belgium/Egypt each miss one full-stat match."
    )
    fig.text(0.08, 0.030, footnote, fontsize=7.8, color="#6b7280", linespacing=1.18)

    plt.tight_layout(rect=[0, 0.105, 1, 1])

    output_path = OUTPUT_FIG_DIR / f"09_progression_quality_{output_suffix}_{phase_label.lower().replace(' ', '_')}.png"
    fig.savefig(output_path, dpi=220, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.show()

    corr = plot_df[["field_tilt_proxy", metric_y]].corr().iloc[0, 1]
    print(f"Saved figure: {output_path}")
    print(f"Correlation between Field Tilt Proxy and {metric_y}: {corr:.3f}")


plot_progression_scatter(
    metric_y="line_break_completion_rate",
    y_label="Line Break Completion Rate (%)",
    title_suffix="Line Break Completion",
    output_suffix="field_tilt_vs_line_break_completion",
    phase_label="Overall",
)

## Visualization 2: Territory vs Defensive Line Break Completion

This chart focuses on the more aggressive progression question: did teams break the opponent's defensive line efficiently?

Completed defensive line breaks are especially useful because they move beyond simple territory gain and point toward deeper access into the opposition structure.


In [ ]:
"""
Step 5: Plot Field Tilt Proxy vs Defensive Line Break Completion Rate
=====================================================================
"""

plot_progression_scatter(
    metric_y="defensive_line_break_completion_rate",
    y_label="Defensive Line Break Completion Rate (%)",
    title_suffix="Defensive Line Break Completion",
    output_suffix="field_tilt_vs_defensive_line_break_completion",
    phase_label="Overall",
)




## Visualization 3: Spain Benchmark Profile

This bar chart summarizes Spain's ranking across progression-quality metrics. It keeps the champion-profile story connected to the wider progression analysis.


In [ ]:
"""
Step 6: Plot Spain's progression-quality ranking profile
========================================================
"""

plot_phase = "Overall"
plot_df = progression_metrics_by_phase[progression_metrics_by_phase["competition_phase"] == plot_phase].copy()

benchmark_metrics = {
    "field_tilt_proxy": "Field Tilt Proxy",
    "line_break_completion_rate": "Line Break Completion Rate",
    "defensive_line_break_completion_rate": "Defensive Line Break Completion Rate",
    "completed_line_breaks_per_match": "Completed Line Breaks per Match",
    "completed_defensive_line_breaks_per_match": "Completed Defensive Line Breaks per Match",
    "defensive_line_break_share": "Defensive Line Break Share",
}

rows = []
for metric, label in benchmark_metrics.items():
    ranks = plot_df[metric].rank(ascending=False, method="min")
    n_teams = plot_df["team_name"].nunique()
    champion_value = plot_df.loc[plot_df["team_name"] == CHAMPION_TEAM, metric].iloc[0]
    champion_rank = int(ranks.loc[plot_df["team_name"] == CHAMPION_TEAM].iloc[0])
    percentile = (n_teams - champion_rank + 1) / n_teams * 100
    rows.append({
        "metric": metric,
        "metric_label": label,
        "champion_value": champion_value,
        "rank": champion_rank,
        "teams_in_phase": n_teams,
        "percentile_rank": percentile,
    })

spain_progression_profile = pd.DataFrame(rows).sort_values("percentile_rank", ascending=True)
profile_path = OUTPUT_DATA_DIR / f"progression_quality_spain_profile_{plot_phase.lower().replace(' ', '_')}.csv"
spain_progression_profile.to_csv(profile_path, index=False, encoding="utf-8-sig")

fig, ax = plt.subplots(figsize=(11.5, 5.8), dpi=150)
fig.patch.set_facecolor("#fbfaf7")
ax.set_facecolor("#fbfaf7")

colors = np.where(spain_progression_profile["percentile_rank"] >= 75, "#047857", np.where(spain_progression_profile["percentile_rank"] >= 50, "#0f766e", "#f59e0b"))

ax.axvspan(0, 50, color="#f3f4f6", alpha=0.55, zorder=0)
ax.axvspan(50, 100, color="#ecfdf5", alpha=0.38, zorder=0)
ax.barh(
    spain_progression_profile["metric_label"],
    spain_progression_profile["percentile_rank"],
    color=colors,
    alpha=0.95,
    height=0.62,
)

for y, (_, row) in enumerate(spain_progression_profile.iterrows()):
    label = f"#{int(row['rank'])} / {int(row['teams_in_phase'])}"
    value = row["percentile_rank"]
    if value >= 72:
        ax.text(value - 2, y, label, va="center", ha="right", fontsize=9, color="white", fontweight="medium")
    else:
        ax.text(value + 1.4, y, label, va="center", ha="left", fontsize=9, color="#111827", fontweight="medium")

ax.axvline(50, linestyle="--", color="#4b5563", linewidth=1.1, alpha=0.80)
ax.text(50, len(spain_progression_profile) - 0.25, "Median", ha="center", va="bottom", fontsize=8.5, color="#4b5563")

ax.set_xlim(0, 108)
ax.set_xlabel("Percentile Rank within Phase (higher = closer to #1)", fontsize=11)
ax.set_title(f"Spain Progression Quality Profile ({plot_phase})", fontsize=15, pad=14)
ax.grid(axis="x", alpha=0.18, color="#9ca3af", linewidth=0.8)
ax.set_axisbelow(True)

for spine in ax.spines.values():
    spine.set_visible(False)

footnote = (
    "Rank #1 means the highest value in the selected phase. Rankings are descriptive, not causal.\n"
    "Source: FIFA Match Centre | Full Official Stats only.\nNote: descriptive rankings; match counts 3-8; Belgium/Egypt each miss one full-stat match."
)
fig.text(0.08, 0.030, footnote, fontsize=7.8, color="#6b7280", linespacing=1.18)

plt.tight_layout(rect=[0, 0.105, 1, 1])

output_path = OUTPUT_FIG_DIR / f"11_progression_quality_spain_profile_{plot_phase.lower().replace(' ', '_')}.png"
fig.savefig(output_path, dpi=220, bbox_inches="tight", facecolor=fig.get_facecolor())
plt.show()

print(f"Saved figure: {output_path}")
print(f"Saved profile: {profile_path}")
display(spain_progression_profile.round(2))





## Draft Interpretation

Use this section as report-writing support. Edit the text after reviewing the charts.


In [ ]:
"""
Step 7: Generate concise draft takeaways
========================================
"""

report_phase = "Overall"
report_df = progression_metrics_by_phase[progression_metrics_by_phase["competition_phase"] == report_phase].copy()

corr_lb = report_df[["field_tilt_proxy", "line_break_completion_rate"]].corr().iloc[0, 1]
corr_dlb = report_df[["field_tilt_proxy", "defensive_line_break_completion_rate"]].corr().iloc[0, 1]

spain_row = report_df[report_df["team_name"] == CHAMPION_TEAM].iloc[0]

print("Progression Quality Draft Notes")
print("=" * 64)
print(f"Phase: {report_phase}")
print(f"Correlation, Field Tilt vs Line Break Completion Rate: {corr_lb:.3f}")
print(f"Correlation, Field Tilt vs Defensive Line Break Completion Rate: {corr_dlb:.3f}")

print("\nSpain profile:")
print(f"- Field Tilt Proxy: {spain_row['field_tilt_proxy']:.2f}%")
print(f"- Line Break Completion Rate: {spain_row['line_break_completion_rate']:.2f}%")
print(f"- Defensive Line Break Completion Rate: {spain_row['defensive_line_break_completion_rate']:.2f}%")
print(f"- Completed Defensive Line Breaks per Match: {spain_row['completed_defensive_line_breaks_per_match']:.2f}")

print("\nSuggested interpretation:")

def describe_correlation(value):
    abs_value = abs(value)
    if abs_value >= 0.70:
        return "strong"
    if abs_value >= 0.40:
        return "moderate"
    return "weak"

lb_strength = describe_correlation(corr_lb)
dlb_strength = describe_correlation(corr_dlb)

print(
    f"In the {report_phase} sample, Field Tilt Proxy has a {lb_strength} descriptive association "
    f"with Line Break Completion Rate (r={corr_lb:.3f}) and a {dlb_strength} descriptive association "
    f"with Defensive Line Break Completion Rate (r={corr_dlb:.3f}). "
    "Spain's champion profile sits near the top of the tournament for Field Tilt, line-break completion, and completed line-break volume. "
    "This describes Spain's progression profile; it does not prove that these metrics caused the title."
)

print("\nCaution:")
print(
    "Line-break metrics describe progression through lines, not shot quality. Because xG is not available in this FIFA dataset, this analysis should not be read as direct chance-quality evaluation. "
    "Team match counts differ from 3 to 8, and Belgium/Egypt each have one full-stat match excluded, so per-match comparisons should be read as descriptive rankings."
)





## Visualization 4: Semifinalists vs Rest of Field

This chart compares the four semifinalists (Spain, Argentina, France, England — identified as the teams with the most matches played) against the remaining 44 teams on territorial dominance and progression quality. It tests whether the pattern observed for Spain individually also holds at the group level.

In [ ]:
"""
Step 7: Semifinalists vs rest — group comparison
==================================================
Semifinalists identified as the 4 teams with the most matches played (8),
consistent with reaching the final and third-place match.
"""

SEMIFINALISTS = ["Spain", "Argentina", "France", "England"]

overall_df = progression_metrics_by_phase[
    progression_metrics_by_phase["competition_phase"] == "Overall"
].copy()

missing_sf = [t for t in SEMIFINALISTS if t not in overall_df["team_name"].values]
if missing_sf:
    raise ValueError(f"Semifinalist teams missing from data: {missing_sf}")

compare_metrics = ["field_tilt_proxy", "line_break_completion_rate"]

overall_df["group"] = np.where(
    overall_df["team_name"].isin(SEMIFINALISTS), "Semifinalists (4)", "Rest of Field (44)"
)

group_comparison = overall_df.groupby("group")[compare_metrics].mean().round(2)
print("[Semifinalists vs Rest — mean values]")
display(group_comparison)

gap = group_comparison.loc["Semifinalists (4)"] - group_comparison.loc["Rest of Field (44)"]
print("\n[Gap: Semifinalists - Rest]")
display(gap.round(2))

fig, axes = plt.subplots(1, 2, figsize=(12, 5), dpi=150)
labels = {"field_tilt_proxy": "Field Tilt Proxy (%)", "line_break_completion_rate": "Line Break Completion Rate (%)"}

for ax, metric in zip(axes, compare_metrics):
    means = group_comparison[metric]
    colors = ["#C9A34E", "#94A3B8"]
    bars = ax.bar(means.index, means.values, color=colors, width=0.55)
    for bar, val in zip(bars, means.values):
        ax.text(bar.get_x() + bar.get_width()/2, val + 1, f"{val:.1f}%", ha="center", fontsize=11, fontweight="bold")
    ax.set_title(labels[metric], fontsize=12)
    ax.set_ylim(0, max(means.values) * 1.25)
    ax.spines[["top", "right"]].set_visible(False)

fig.suptitle("Semifinalists Separated Early — on Territory and Progression", fontsize=15, y=1.03)
fig.text(0.5, -0.02,
    "Semifinalists = Spain, Argentina, France, England (most matches played, 8 each).\n"
    "Source: FIFA Match Centre | Full Official Stats only.",
    ha="center", fontsize=8, color="gray")
plt.tight_layout()

semifinalist_chart_path = OUTPUT_DATA_DIR / "semifinalists_vs_rest_territory_progression.png"
plt.savefig(semifinalist_chart_path, dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved: {semifinalist_chart_path}")